# Phase 4 — Review NLP

Phase 3 established *what*: deeper discounts predict lower ratings / rating volume after controls.
Phase 4 finds *why*: what heavily discounted, lower-rated products' reviews cluster around — by topic and category — for Phase 5.


## Block A — Load and merge


In [ ]:
import pandas as pd
from scipy import stats

from config import (
    REVIEW_TEXT_PATH,
    NLP_RESULTS_PATH,
    TOPIC_SUMMARY_PATH,
    TOPIC_SUMMARY_ACTIONABLE_PATH,
    NLP_OUTLIER_XTAB_PATH,
    SENTIMENT_MODEL,
    EMBEDDING_MODEL,
    BERTOPIC_MIN_TOPIC_SIZE,
    BERTOPIC_MODEL_DIR,
    BERTOPIC_TOPICS_PATH,
    BERTOPIC_META_PATH,
    ARTIFACTS_DIR,
    RANDOM_SEED,
    TOPIC_MIN_PRODUCTS,
    CRITICAL_RATING_QUANTILE,
    CRITICAL_DISCOUNT_QUANTILE,
    NLP_DOMAIN_STOPWORDS,
)
from utils import load_star_schema

review_text = pd.read_csv(REVIEW_TEXT_PATH)
metrics_df = load_star_schema()  # raw category_main (not Phase-3 collapsed)

text_df = review_text.merge(metrics_df, on="product_id", how="inner")
assert len(text_df) == len(metrics_df), (
    f"Merge dropped rows: {len(text_df)} vs {len(metrics_df)}"
)

text_df["full_text"] = (
    text_df["review_title"].fillna("") + ". " + text_df["review_content"].fillna("")
)
print(text_df.shape)
print(text_df["full_text"].str.len().describe())


## Block B — Transformer star-sentiment (batched)


In [4]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    truncation=True,
    max_length=512,
    batch_size=32,
)

texts = text_df["full_text"].fillna("").astype(str).tolist()
raw = sentiment_pipe(texts)

text_df["predicted_stars"] = [int(r["label"][0]) for r in raw]
text_df["prediction_confidence"] = [r["score"] for r in raw]

print(text_df[["predicted_stars", "prediction_confidence"]].describe())


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

       predicted_stars  prediction_confidence
count      1350.000000            1350.000000
mean          3.514815               0.541032
std           0.904319               0.135427
min           1.000000               0.220548
25%           3.000000               0.449002
50%           4.000000               0.515191
75%           4.000000               0.615281
max           5.000000               0.986646


## Block C — Trust gate (validate vs product rating)

Stop here if Pearson is weak or MAE is large before running topic modeling.


In [5]:
mae = (text_df["predicted_stars"] - text_df["rating"]).abs().mean()
exact_match_rate = (text_df["predicted_stars"] == text_df["rating"].round()).mean()
r_p, p_p = stats.pearsonr(text_df["predicted_stars"], text_df["rating"])
r_s, p_s = stats.spearmanr(text_df["predicted_stars"], text_df["rating"])

print(f"MAE: {mae:.3f} stars | Exact match: {exact_match_rate:.1%}")
print(f"Pearson r={r_p:.3f} (p={p_p:.4f})  Spearman r={r_s:.3f} (p={p_s:.4f})")


MAE: 0.757 stars | Exact match: 46.4%
Pearson r=0.416 (p=0.0000)  Spearman r=0.369 (p=0.0000)


## Block D - Topic modeling (BERTopic + seeded UMAP)

After fit, topic labels:
- **API key set** (`OPENAI_API_KEY` in `.env`): LLM plain-English label (cached)
- **Empty / missing key, offline, or API error:** top-term concatenation

Fitted BERTopic is saved under `python/artifacts/`; re-runs load from disk and skip embed/UMAP/cluster when meta matches.


In [ ]:
import json

import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from topic_labeling import extract_bertopic_terms, label_topics

nlp_stop_words = list(ENGLISH_STOP_WORDS.union(NLP_DOMAIN_STOPWORDS))
texts = text_df["full_text"].tolist()

_bertopic_meta = {
    "embedding_model": EMBEDDING_MODEL,
    "min_topic_size": BERTOPIC_MIN_TOPIC_SIZE,
    "random_seed": RANDOM_SEED,
    "n_docs": len(texts),
}


def _bertopic_cache_valid() -> bool:
    if not (
        BERTOPIC_MODEL_DIR.is_dir()
        and BERTOPIC_TOPICS_PATH.is_file()
        and BERTOPIC_META_PATH.is_file()
    ):
        return False
    try:
        prev = json.loads(BERTOPIC_META_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    if prev != _bertopic_meta:
        return False
    topics_arr = np.load(BERTOPIC_TOPICS_PATH)
    return len(topics_arr) == len(texts)


if _bertopic_cache_valid():
    print(f"Loading fitted BERTopic from {BERTOPIC_MODEL_DIR}")
    topic_model = BERTopic.load(str(BERTOPIC_MODEL_DIR))
    topics = np.load(BERTOPIC_TOPICS_PATH).tolist()
    method_used = (
        f"BERTopic ({EMBEDDING_MODEL}, seeded UMAP, stopword vectorizer) [cached]"
    )
else:
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_SEED,
    )
    # Stopwords affect c-TF-IDF keywords only - clustering uses embeddings + UMAP + HDBSCAN
    vectorizer_model = CountVectorizer(stop_words=nlp_stop_words, min_df=2)

    topic_model = BERTopic(
        embedding_model=embedder,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=BERTOPIC_MIN_TOPIC_SIZE,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(texts)

    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    topic_model.save(
        str(BERTOPIC_MODEL_DIR),
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBEDDING_MODEL,
    )
    np.save(BERTOPIC_TOPICS_PATH, np.asarray(topics))
    BERTOPIC_META_PATH.write_text(
        json.dumps(_bertopic_meta, indent=2), encoding="utf-8"
    )
    print(f"Saved fitted BERTopic -> {BERTOPIC_MODEL_DIR}")
    method_used = f"BERTopic ({EMBEDDING_MODEL}, seeded UMAP, stopword vectorizer)"

text_df["topic"] = topics
print(topic_model.get_topic_info().head(15))

print(f"\nUsed: {method_used}")
print("Topic value counts:\n", text_df["topic"].value_counts().head(20))
print(f"Outlier share (topic=-1): {(text_df['topic'] == -1).mean():.1%}")

top_terms_by_topic = extract_bertopic_terms(topic_model)
for tid in text_df["topic"].unique():
    top_terms_by_topic.setdefault(int(tid), [])
dim_topic = label_topics(top_terms_by_topic)
print(dim_topic.head(10))


## Block E - Category x topic summary + watchlist / Critical export

**Watchlist** (relative): n >= TOPIC_MIN_PRODUCTS and below-category-median rating + above-category-median discount.

**Critical** (Page 1 KPI): among watchlist cells, rating <= Q1 and discount >= median of that watchlist.


In [ ]:
# Exclude BERTopic outliers (-1) from topic tables; keep them in nlp_results
summary_df = text_df.loc[text_df["topic"] != -1].copy()

topic_summary = (
    summary_df.groupby(["category_main", "topic"], observed=True)
    .agg(
        n_products=("product_id", "count"),
        avg_rating=("rating", "mean"),
        avg_discount=("discount_percentage", "mean"),
        avg_predicted_stars=("predicted_stars", "mean"),
    )
    .reset_index()
)

# Watchlist: relative medians within category
topic_summary["is_watchlist"] = (
    (topic_summary["avg_rating"]
     <= topic_summary.groupby("category_main")["avg_rating"].transform("median"))
    & (topic_summary["avg_discount"]
       >= topic_summary.groupby("category_main")["avg_discount"].transform("median"))
)

topic_summary_actionable = topic_summary.loc[
    (topic_summary["n_products"] >= TOPIC_MIN_PRODUCTS)
    & topic_summary["is_watchlist"]
].copy()

# Critical: stricter cut among watchlist cells (Page 1 KPI)
r_cut = topic_summary_actionable["avg_rating"].quantile(CRITICAL_RATING_QUANTILE)
d_cut = topic_summary_actionable["avg_discount"].quantile(CRITICAL_DISCOUNT_QUANTILE)
topic_summary_actionable["is_critical"] = (
    (topic_summary_actionable["avg_rating"] <= r_cut)
    & (topic_summary_actionable["avg_discount"] >= d_cut)
)
# Every actionable row is already watchlist — drop redundant column
topic_summary_actionable = (
    topic_summary_actionable.drop(columns=["is_watchlist"])
    .sort_values(["avg_rating", "avg_discount"])
)

print(topic_summary_actionable.to_string(index=False))
print(f"\nmethod_used={method_used}")
n_crit = int(topic_summary_actionable["is_critical"].sum())
print(
    f"watchlist cells (n>={TOPIC_MIN_PRODUCTS}): {len(topic_summary_actionable)} "
    f"| critical cells: {n_crit} "
    f"| critical products: {int(topic_summary_actionable.loc[topic_summary_actionable.is_critical, 'n_products'].sum())}"
)

text_df[["product_id", "predicted_stars", "prediction_confidence", "topic"]].to_csv(
    NLP_RESULTS_PATH, index=False
)
topic_summary.sort_values(["category_main", "avg_rating"]).to_csv(
    TOPIC_SUMMARY_PATH, index=False
)
topic_summary_actionable.to_csv(TOPIC_SUMMARY_ACTIONABLE_PATH, index=False)
print(
    f"Wrote {NLP_RESULTS_PATH.name}, {TOPIC_SUMMARY_PATH.name}, "
    f"{TOPIC_SUMMARY_ACTIONABLE_PATH.name}"
)


## Step 5.0 — Power BI export tables

Builds `fact_product_analytics.csv` with `is_watchlist_product` / `is_critical_product`,
and syncs `dim_topic.csv` + actionable `topic_label`.

Safe to re-run after Phase 4:
`python/5_build_powerbi_exports.py`


In [ ]:
import runpy
from config import PROJECT_ROOT

# Assign to _ so Jupyter does not dump the entire runpy namespace as cell output.
_ = runpy.run_path(
    str(PROJECT_ROOT / "python" / "5_build_powerbi_exports.py"),
    run_name="__main__",
)


## Block F — Outlier bucket check (topic = -1)

Does the Phase 3 story (high discount / low rating) over-index in unclustered reviews?


In [9]:
text_df["is_outlier"] = text_df["topic"] == -1
text_df["discount_q"] = pd.qcut(
    text_df["discount_percentage"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"]
)
text_df["rating_band"] = pd.cut(
    text_df["rating"],
    bins=[0, 3.5, 4.0, 4.5, 5.01],
    labels=["<=3.5", "3.5-4.0", "4.0-4.5", "4.5-5.0"],
    include_lowest=True,
)

outlier_by_discount = text_df.groupby("discount_q", observed=True)["is_outlier"].agg(
    outlier_rate="mean", n="size"
)
outlier_by_rating = text_df.groupby("rating_band", observed=True)["is_outlier"].agg(
    outlier_rate="mean", n="size"
)
outlier_xtab = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)["is_outlier"]
    .mean()
    .unstack("rating_band")
)
outlier_xtab_n = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)["is_outlier"]
    .size()
    .unstack("rating_band")
)

print(f"Overall outlier rate: {text_df['is_outlier'].mean():.1%} (n={len(text_df)})")
print("\nOutlier rate by discount quartile:\n", outlier_by_discount.round(3).to_string())
print("\nOutlier rate by rating band:\n", outlier_by_rating.round(3).to_string())
print("\nOutlier rate (discount_q x rating_band):\n", outlier_xtab.round(3).to_string())
print("\nCell sizes (discount_q x rating_band):\n", outlier_xtab_n.to_string())

# Long-form export for Phase 5 (rate + n per cell)
outlier_export = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)
    .agg(outlier_rate=("is_outlier", "mean"), n=("is_outlier", "size"))
    .reset_index()
)
outlier_export.to_csv(NLP_OUTLIER_XTAB_PATH, index=False)
print(f"\nWrote {NLP_OUTLIER_XTAB_PATH.name}")


Overall outlier rate: 7.6% (n=1350)

Outlier rate by discount quartile:
             outlier_rate    n
discount_q                   
Q1_low             0.080  349
Q2                 0.124  339
Q3                 0.034  327
Q4_high            0.063  335

Outlier rate by rating band:
              outlier_rate    n
rating_band                   
<=3.5               0.090   67
3.5-4.0             0.111  432
4.0-4.5             0.055  823
4.5-5.0             0.107   28

Outlier rate (discount_q x rating_band):
 rating_band  <=3.5  3.5-4.0  4.0-4.5  4.5-5.0
discount_q                                   
Q1_low       0.000    0.148    0.056    0.167
Q2           0.071    0.193    0.090    0.000
Q3           0.045    0.038    0.031    0.000
Q4_high      0.160    0.067    0.038    0.250

Cell sizes (discount_q x rating_band):
 rating_band  <=3.5  3.5-4.0  4.0-4.5  4.5-5.0
discount_q                                   
Q1_low           6       88      249        6
Q2              14      119     

## Findings

- **Sentiment trust gate:** MAE = 0.757 stars | Exact match = 46.4% | Pearson r = 0.416 | Spearman r = 0.369. Gate passed.
- **Topic method:** `BERTopic (all-MiniLM-L6-v2, seeded UMAP, stopword vectorizer)`; fitted model cached under `python/artifacts/` (re-runs skip embed/UMAP/cluster). Outlier share ~ **7.6%** (102 products); kept in `nlp_results.csv`, excluded from topic summaries.
- **Topic labels:** LLM JSON when `OPENAI_API_KEY` is set in `.env` (cached); otherwise top-term concatenation.
- **Watchlist vs Critical (Phase 5):**
  - **Watchlist** — relative category-median flag, `n>=10`: **14 cells / 619 products** (~46%). Exploration list only — not "objectively bad."
  - **Critical** — among watchlist, rating <= Q1 and discount >= median: **2 cells / 133 products** (~10%). Use this for the Page 1 KPI.
  - Critical themes today: Electronics remotes (~3.84*, 55% disc) and earbuds (~3.94*, 57% disc).
- **Outputs for Phase 5:**
  - `nlp_results.csv` — per-product stars + topic
  - `topic_summary.csv` — all category x topic cells (+ `is_watchlist`)
  - `topic_summary_actionable.csv` — watchlist cells + `is_critical` + `topic_label`
  - `nlp_outlier_xtab.csv` — outlier rate (+ n) by discount quartile x rating band
  - `fact_product_analytics.csv` — `is_watchlist_product`, `is_critical_product`
  - `dim_topic.csv` / `topic_label_cache.csv` — topic_id -> topic_label
- **Outlier check:** overall ~ 7.6%. Deepest-discount quartile is **not** elevated. Phase 3 signal lives in named topics, not `-1`.
- **Implication for Phase 5.1:** Page 1 card = **Critical (133)**; Page 3 table = full **Watchlist (619)** with relative-median tooltip.
